In [1]:
import numpy as np
import pandas as pd
from flaml import AutoML
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")



In [ ]:



FEATURE_COLS = [
    "DESI_FASTSPEC_LOGMSTAR",
    "DESI_FASTSPEC_SFR",
    "DESI_FASTSPEC_VDISP",
    "DESI_FASTSPEC_DN4000",
    "DESI_FASTSPEC_g_minus_r",
    "DESI_FASTSPEC_AGE",
    "ZTF_x1",
    "ZTF_c",
]

X = df[FEATURE_COLS]
y = df["residuals"]
w = 1.0 / df["ZTF_sigma_mu"] ** 2

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=42
)

print(f"{len(df)} SNe Ia, {len(FEATURE_COLS)} features")

automl = AutoML()

automl.fit(
    X_train, y_train,
    sample_weight=w_train.values,
    task="regression",
    metric="rmse",
    time_budget=600,
    estimator_list=[
        "rf",
        "xgboost",
        "extra_tree",
        "enet",
    ],
    eval_method="cv",
    n_splits=5,
    seed=42,
    verbose=2,
)

print(f"\nBest model:   {automl.best_estimator}")
print(f"Best CV RMSE: {-automl.best_loss:.5f}")
print(f"Best config:  {automl.best_config}")


def nmad(residuals):
    return 1.4826 * np.median(np.abs(residuals - np.median(residuals)))

# FLAML stores the best model per estimator type
results = []
for est_name, est_info in automl.best_config_per_estimator.items():
    if est_info is None:
        continue
    model = automl.best_model_for_estimator(est_name)
    y_pred = model.predict(X_test)
    resid = y_test - y_pred
    results.append({
        "Model": est_name,
        "Test RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "Test MAE": mean_absolute_error(y_test, y_pred),
        "Test R²": r2_score(y_test, y_pred),
        "Test NMAD": nmad(resid),
    })

df_results = pd.DataFrame(results).sort_values("Test RMSE")
print(df_results.to_string(index=False, float_format="%.4f"))

521 SNe Ia, 8 features


KeyboardInterrupt: 

In [2]:
from pycaret.regression import *


import numpy as np
import pandas as pd
from flaml import AutoML
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = "data/ZTF_DESI_matched_lc_cuts_z_cuts_hostprop_cleaned_tripp_uncorrected.csv"
df = pd.read_csv(DATA_PATH)

FEATURE_COLS = [
    "DESI_FASTSPEC_LOGMSTAR",
    "DESI_FASTSPEC_SFR",
    "DESI_FASTSPEC_VDISP",
    "DESI_FASTSPEC_DN4000",
    "DESI_FASTSPEC_g_minus_r",
    "DESI_FASTSPEC_AGE",
    "ZTF_x1",
    "ZTF_c",
]

X = df[FEATURE_COLS]
y = df["residuals"]
w = 1.0 / df["ZTF_sigma_mu"] ** 2

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, y, w, test_size=0.2, random_state=42
)

print(f"{len(df)} SNe Ia, {len(FEATURE_COLS)} features")

df_pycaret = df[FEATURE_COLS + ["residuals"]].copy()

setup(df_pycaret, target="residuals", session_id=42, fold=5, verbose=False)

best = compare_models(
    include=["ridge", "en", "rf", "xgboost", "et", "mlp"],
    sort="RMSE",
)

513 SNe Ia, 8 features


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,0.1498,0.0408,0.2005,0.6419,0.1294,2.7294,0.2320
ridge,Ridge Regression,0.1485,0.0408,0.2009,0.6497,0.1334,2.6757,0.4280
rf,Random Forest Regressor,0.1534,0.0425,0.2048,0.6292,0.1334,3.0406,0.2280
mlp,MLP Regressor,0.1542,0.0430,0.2065,0.6279,0.1361,2.6565,0.0180
xgboost,Extreme Gradient Boosting,0.1620,0.0492,0.2213,0.5585,0.1410,3.0526,0.2380
en,Elastic Net,0.2660,0.1186,0.3420,-0.0041,0.1804,3.4004,0.2120


In [3]:
best = compare_models(
    include=["ridge", "en", "rf", "xgboost", "et"],
    sort="RMSE",
    n_select=5,
)

tuned = [tune_model(m, optimize="RMSE", n_iter=100, search_library="optuna") for m in best]

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,0.1498,0.0408,0.2005,0.6419,0.1294,2.7294,0.0160
ridge,Ridge Regression,0.1485,0.0408,0.2009,0.6497,0.1334,2.6757,0.0060
rf,Random Forest Regressor,0.1534,0.0425,0.2048,0.6292,0.1334,3.0406,0.0220
xgboost,Extreme Gradient Boosting,0.1620,0.0492,0.2213,0.5585,0.1410,3.0526,0.0140
en,Elastic Net,0.2660,0.1186,0.3420,-0.0041,0.1804,3.4004,0.2640


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1426,0.0370,0.1924,0.7247,0.1171,0.5981
1,0.1565,0.0448,0.2118,0.6697,0.1346,1.2031
2,0.1399,0.0372,0.1929,0.5417,0.1283,9.1818
3,0.1248,0.0269,0.1640,0.7030,0.1049,1.0140
4,0.1681,0.0525,0.2292,0.6464,0.1573,2.3051
Mean,0.1464,0.0397,0.1981,0.6571,0.1284,2.8604
Std,0.0148,0.0086,0.0218,0.0637,0.0176,3.2107


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1400,0.0335,0.1832,0.7506,0.1161,0.6003
1,0.1423,0.0420,0.2048,0.6910,0.1235,0.7286
2,0.1338,0.0344,0.1854,0.5768,0.1316,8.2994
3,0.1220,0.0260,0.1611,0.7133,0.1097,0.8815
4,0.1626,0.0476,0.2181,0.6798,0.1490,2.3850
Mean,0.1401,0.0367,0.1905,0.6823,0.1260,2.5790
Std,0.0133,0.0074,0.0195,0.0580,0.0137,2.9319


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1495,0.0402,0.2004,0.7014,0.1239,0.5714
1,0.1542,0.0422,0.2054,0.6894,0.1322,1.3126
2,0.1465,0.0408,0.2019,0.4978,0.1310,10.1766
3,0.1287,0.0275,0.1658,0.6966,0.1105,1.0776
4,0.1698,0.0518,0.2277,0.6509,0.1544,2.0027
Mean,0.1498,0.0405,0.2002,0.6472,0.1304,3.0282
Std,0.0132,0.0078,0.0199,0.0768,0.0143,3.6038


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1454,0.0352,0.1877,0.7382,0.1172,0.5725
1,0.1576,0.0419,0.2046,0.6917,0.1295,1.1263
2,0.1525,0.0433,0.2081,0.4667,0.1420,8.4540
3,0.1333,0.0307,0.1751,0.6614,0.1220,1.0817
4,0.1755,0.0542,0.2329,0.6348,0.1559,2.3862
Mean,0.1529,0.0411,0.2017,0.6386,0.1333,2.7241
Std,0.0140,0.0080,0.0196,0.0925,0.0141,2.9266


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.1437,0.0353,0.1879,0.7374,0.1193,0.6121
1,0.1416,0.0417,0.2043,0.6927,0.1250,0.6518
2,0.1319,0.0339,0.1840,0.5829,0.1296,8.2880
3,0.1207,0.0263,0.1623,0.7092,0.1114,0.8993
4,0.1662,0.0481,0.2193,0.6763,0.1480,2.4825
Mean,0.1408,0.0371,0.1916,0.6797,0.1267,2.5867
Std,0.0151,0.0074,0.0193,0.0525,0.0123,2.9328


In [4]:
for model in tuned:
    name = type(model).__name__
    params = model.get_params()
    print(f"\n{'='*50}")
    print(f"{name}")
    for k, v in params.items():
        print(f"  {k}: {v}")


ExtraTreesRegressor
  bootstrap: False
  ccp_alpha: 0.0
  criterion: squared_error
  max_depth: 7
  max_features: 0.9671460926875693
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 2.9742585807574298e-09
  min_samples_leaf: 3
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 196
  n_jobs: -1
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False

Ridge
  alpha: 0.03485892649418611
  copy_X: True
  fit_intercept: False
  max_iter: None
  positive: False
  random_state: 42
  solver: auto
  tol: 0.0001

RandomForestRegressor
  bootstrap: True
  ccp_alpha: 0.0
  criterion: squared_error
  max_depth: 6
  max_features: 0.9791213870235925
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 3.2425786690433903e-09
  min_samples_leaf: 4
  min_samples_split: 7
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 228
  n_jobs: -1
  oob_score: False
  random_state: 42
  verbose: 0
  warm

In [2]:
import numpy as np
import pandas as pd
from flaml import AutoML
from sklearn.model_selection import train_test_split, cross_val_predict, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings("ignore")



DATA_PATH = "data/ZTF_DESI_matched_lc_cuts_z_cuts_hostprop_cleaned.csv"
df = pd.read_csv(DATA_PATH)

print(df["residuals"].std())
print(df["residuals"].describe())

0.1863875197809497
count    521.000000
mean      -0.019251
std        0.186388
min       -0.629900
25%       -0.123732
50%       -0.005231
75%        0.101271
max        0.583876
Name: residuals, dtype: float64


In [4]:
from scipy.stats import spearmanr

rho_x1, p_x1 = spearmanr(df["ZTF_x1"], df["residuals"])
rho_c, p_c = spearmanr(df["ZTF_c"], df["residuals"])
print(f"x1 vs residuals: ρ={rho_x1:.3f}, p={p_x1:.2e}")
print(f"c  vs residuals: ρ={rho_c:.3f}, p={p_c:.2e}")

x1 vs residuals: ρ=0.134, p=2.19e-03
c  vs residuals: ρ=-0.276, p=1.48e-10
